# 19_Packet_P002_Variance_Reduction
## Materia Arche Agent OS v3.0

### Correction Notice (NB18 / P-001)
NB18 footer stated "Model is confirmed robust for Phase 2 candidates" despite the status being Promising/Iterate (std 0.0418 > 0.03 threshold). This violated Law #7: No claim above earned evidence level. NB18 has been corrected.

---

**Work Packet ID:** P-002
**Decision this packet informs:** Can we reduce tau-b variance below 0.03 while maintaining mean > 0.25?
**Fastest falsifier:** Test 3 variance reduction strategies on the same 20-split framework — if none reduce std below 0.03, the current model variance is structural.
**Success threshold:** At least one strategy achieves mean tau-b > 0.25 and std < 0.03
**Failure / kill threshold:** No strategy reduces std below 0.035
**Reuse requirement if it fails:** Document that variance is structural for this dataset size; recommend accepting std ~0.04 or collecting more data
**Dataset ID/version:** perovskite_stability_clean.csv (1,543 devices)
**Benchmark ID:** P-001 locked ET (mean tau-b 0.3025, std 0.0418)
**Owner:** ML Scientist
**Reviewer:** Evidence Guardian
**Release ceiling:** Internal

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded — Packet P-002")

Libraries loaded — Packet P-002


## 1. Load data and reproduce P-001 baseline

In [2]:
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices")

all_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
                'MA', 'FA', 'Cs',
                'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
                'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
                'JV_default_Jsc', 'JV_default_FF']

# Top 8 SHAP features from NB17
top8_features = ['JV_default_Jsc', 'Perovskite_band_gap', 'JV_default_Voc',
                 'JV_default_FF', 'Cell_area_measured', 'FA',
                 'first_Prvskt_annealing_temperature', 'Perovskite_thickness']

X_all = df[all_features].fillna(0)
X_top8 = df[top8_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

# Locked ET config from NB15
et_params = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False)

print(f"All features: {len(all_features)}")
print(f"Top-8 SHAP features: {len(top8_features)}")

Dataset: 1543 devices
All features: 16
Top-8 SHAP features: 8


## 2. Define evaluation framework

Same 20-split framework as P-001 for direct comparison.

In [3]:
n_splits = 20
seeds = list(range(1, n_splits + 1))

def evaluate_strategy(X, y, model_fn, seeds):
    """Run model across 20 random splits, return tau-b array."""
    taus = []
    for seed in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
        model = model_fn()
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        tau, _ = kendalltau(y_te, pred)
        taus.append(tau)
    return np.array(taus)

print("Evaluation framework ready")

Evaluation framework ready


## 3. Test variance reduction strategies

**Strategy A:** Locked ET, all 16 features (P-001 baseline)
**Strategy B:** Locked ET, top-8 SHAP features only (reduce noise)
**Strategy C:** ET with bootstrap=True (bagging adds variance reduction)
**Strategy D:** ET with more trees (n_estimators=1500)

In [4]:
# Strategy A: P-001 baseline (locked ET, all 16 features)
print("Running Strategy A: Locked ET, 16 features...")
taus_A = evaluate_strategy(X_all, y,
    lambda: ExtraTreesRegressor(random_state=42, **et_params), seeds)

# Strategy B: Top-8 SHAP features
print("Running Strategy B: Locked ET, top-8 SHAP features...")
taus_B = evaluate_strategy(X_top8, y,
    lambda: ExtraTreesRegressor(random_state=42, **et_params), seeds)

# Strategy C: ET with bootstrap=True
print("Running Strategy C: ET with bootstrap=True...")
et_bootstrap_params = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                           min_samples_leaf=1, bootstrap=True)
taus_C = evaluate_strategy(X_all, y,
    lambda: ExtraTreesRegressor(random_state=42, **et_bootstrap_params), seeds)

# Strategy D: More trees (1500)
print("Running Strategy D: ET with 1500 trees...")
et_1500_params = dict(n_estimators=1500, max_features='sqrt', min_samples_split=3,
                      min_samples_leaf=1, bootstrap=False)
taus_D = evaluate_strategy(X_all, y,
    lambda: ExtraTreesRegressor(random_state=42, **et_1500_params), seeds)

print("All strategies complete.")

Running Strategy A: Locked ET, 16 features...


Running Strategy B: Locked ET, top-8 SHAP features...


Running Strategy C: ET with bootstrap=True...


Running Strategy D: ET with 1500 trees...


All strategies complete.


## 4. Results comparison

In [5]:
strategies = {
    'A: Locked ET (16 feat)': taus_A,
    'B: Top-8 SHAP feat': taus_B,
    'C: Bootstrap=True': taus_C,
    'D: 1500 trees': taus_D,
}

print("=" * 75)
print("VARIANCE REDUCTION — 20 RANDOM SPLITS")
print("=" * 75)
print(f"{'Strategy':<25} {'Mean tau-b':>10} {'Std':>8} {'Min':>8} {'Max':>8} {'Pass?':>8}")
print("-" * 75)

results_rows = []
best_strategy = None
best_std = 1.0

for name, taus in strategies.items():
    m, s = np.mean(taus), np.std(taus)
    passed = "YES" if m > 0.25 and s < 0.03 else "no"
    print(f"{name:<25} {m:>10.4f} {s:>8.4f} {np.min(taus):>8.4f} {np.max(taus):>8.4f} {passed:>8}")
    results_rows.append({
        'Strategy': name,
        'Mean_tau_b': round(m, 4),
        'Std_tau_b': round(s, 4),
        'Min': round(np.min(taus), 4),
        'Max': round(np.max(taus), 4),
        'Pass': passed
    })
    if s < best_std:
        best_std = s
        best_strategy = name

print("")
print(f"Lowest variance: {best_strategy} (std {best_std:.4f})")
print(f"Target: std < 0.03")

results_df = pd.DataFrame(results_rows)
results_df.to_csv("Packet_P002_Results.csv", index=False)
print("\nPacket_P002_Results.csv exported")

VARIANCE REDUCTION — 20 RANDOM SPLITS
Strategy                  Mean tau-b      Std      Min      Max    Pass?
---------------------------------------------------------------------------
A: Locked ET (16 feat)        0.3025   0.0418   0.2067   0.3587       no
B: Top-8 SHAP feat            0.2740   0.0364   0.1937   0.3212       no
C: Bootstrap=True             0.2953   0.0404   0.2015   0.3463       no
D: 1500 trees                 0.3028   0.0411   0.2087   0.3582       no

Lowest variance: B: Top-8 SHAP feat (std 0.0364)
Target: std < 0.03

Packet_P002_Results.csv exported


## 5. Honest status footer

In [6]:
# Determine status based on whether any strategy passed
any_passed = any(r['Pass'] == 'YES' for r in results_rows)
any_improved = best_std < 0.035  # below kill threshold

if any_passed:
    status = "Confirmed"
    decision = "Advance"
elif any_improved:
    status = "Promising"
    decision = "Iterate"
else:
    status = "Negative"
    decision = "Stop"

print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Packet ID: P-002")
print(f"Status: {status}")
print(f"Evidence level: E2 (benchmarked comparison)")
print(f"Decision outcome: {decision}")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"Benchmark: P-001 locked ET (mean 0.3025, std 0.0418)")
print(f"This run: {len(strategies)} strategies tested across 20 splits each")
print(f"Best variance: {best_strategy} (std {best_std:.4f})")

if any_passed:
    print(f"What worked: Variance reduced below 0.03 threshold")
    print(f"What failed or remains uncertain: Need to verify on held-out data")
    print(f"What changes next: Adopt winning strategy as new locked config")
elif any_improved:
    print(f"What worked: Some variance reduction achieved")
    print(f"What failed or remains uncertain: Still above 0.03 threshold")
    print(f"What changes next: Consider accepting higher variance or collecting more data")
else:
    print(f"What worked: Confirmed variance is structural, not tunable")
    print(f"What failed or remains uncertain: No strategy reduced std below 0.035")
    print(f"What changes next: Accept std ~0.04 as structural for 1,543 devices, or collect more data")

print(f"Reusable asset created: Packet_P002_Results.csv")
print(f"Safeguard added: Tested 4 strategies before accepting variance as structural")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Packet ID: P-002
Status: Negative
Evidence level: E2 (benchmarked comparison)
Decision outcome: Stop
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)
Benchmark: P-001 locked ET (mean 0.3025, std 0.0418)
This run: 4 strategies tested across 20 splits each
Best variance: B: Top-8 SHAP feat (std 0.0364)
What worked: Confirmed variance is structural, not tunable
What failed or remains uncertain: No strategy reduced std below 0.035
What changes next: Accept std ~0.04 as structural for 1,543 devices, or collect more data
Reusable asset created: Packet_P002_Results.csv
Safeguard added: Tested 4 strategies before accepting variance as structural

Reviewer sign-off: Evidence Guardian __________
